# Chapter 14 — Vision-Language Models

Chapter 13's CLIP can tell you *how well* a sentence matches a picture. It
cannot tell you anything else, and the reason is structural: CLIP is two
encoders and nothing that generates. Ask it "what is in this image?" and there
is no mechanism to answer, because nowhere in the architecture is there a
decoder producing tokens.

This chapter adds one. The result is the **LLaVA** recipe (Liu et al., 2023),
and its claim is startling enough to be worth stating before we build it:

> Take a frozen vision encoder. Take a frozen language model. Train **one small
> MLP** between them that turns image features into vectors the language model
> will accept as if they were words. That is the whole model.

No new attention mechanism, no joint architecture, no retraining of either
tower. The image becomes a handful of extra *tokens* prepended to the prompt,
and the language model, which has never seen a picture, reads them the way it
reads any other context.

We are going to build that, and then test whether the claim survives contact
with measurement. Three things need checking, and only the first is obvious:

1. Does the model actually **use the image**, or is it just reciting the caption
   prior? (Module 4 builds a blind baseline to find out.)
2. How many image tokens do you need?
3. Does freezing the language model really work, and *what exactly* is doing
   the work when it does?

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | The architecture, and a frozen CLIP tower recovered from Chapter 13 | `zalando-datasets/fashion_mnist` |
| 2 | A **causal language model**: Chapter 10's block with Chapter 9's causal mask | — |
| 3 | The **projector**: image features as tokens in the LM's embedding space | — |
| 4 | Training the bridge, generating captions, and the **blind baseline** | — |
| 5 | The LLaVA ablation: what must be pretrained, and how little must be trained | — |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~2 minutes on a GPU. Two short pretraining runs (the vision tower
and the language model, under ten seconds each), then seven VLM trainings: the
model itself, its blind twin, and five for the ablations. Knobs are marked
`# <- knob`.


In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (7, 4.5)
print("torch", torch.__version__, "| device:", device)

---
# Module 1 — The Architecture

## 1.1 What has to be true for this to work

🧠 **The intuition.** A language model's first layer is an embedding table: it
turns token id 4,382 into a vector, and every layer above it only ever sees
vectors. It has no idea those vectors came from a lookup table. So if you can
produce a vector *of the right shape, living in roughly the right region of
space*, you can hand it to the model and it will process it like a word.

That is the entire trick. An image encoder produces feature vectors; a small
learned MLP maps them into the language model's embedding space; those become
extra tokens at the front of the sequence. The language model then does what it
always does, attending over its context and predicting the next token, except
that part of its context is a photograph.

📐 **The math.** With a frozen vision encoder $f_V$, a projector $P_\theta$, and
a language model $g_\phi$ over embedding table $E$:

$$ \underbrace{P_\theta\big(f_V(x)\big)}_{k \times d_{\text{lm}} \;\text{"image tokens"}} \;\Vert\; \underbrace{E[t_1], \dots, E[t_m]}_{\text{ordinary word embeddings}} \;\longrightarrow\; g_\phi \;\longrightarrow\; p(t_{i+1} \mid \cdot) $$

The loss is plain next-token cross-entropy on the caption, and, importantly,
it is only applied to the **text** positions. Nothing asks the model to predict
image tokens; they are context, not targets.

**Two ways to get the image in.** Prepending tokens (above) is LLaVA's choice
and is the simpler one: the sequence just gets longer, and self-attention
already knows how to mix a longer sequence. The alternative is
**cross-attention**: keep the text sequence its original length, and let each
block additionally attend to image features with $K, V$ taken from the vision
side. That is Flamingo's design, and it is the same modification you met at the
end of Chapter 12 for text-conditioned diffusion. Prefix tokens cost you
$O((k+m)^2)$ attention; cross-attention costs $O(k \cdot m)$ but needs new
parameters inside every block. We build the prefix version.

💻 **The code.** First the data: Chapter 13's images and captions, unchanged.


In [ ]:
fashion = load_dataset("zalando-datasets/fashion_mnist")
CLASS_NAMES = [c.replace(" - ", "-").replace(" / ", "/") for c in fashion["train"].features["label"].names]
N_TRAIN, N_TEST = 20_000, 5_000            # <- knob

def to_arrays(split, n):
    ds = split.shuffle(seed=42).select(range(n))
    X = np.stack([np.array(im) for im in ds["image"]]).astype("float32") / 255.0
    return X, np.array(ds["label"])

Xtr, ytr = to_arrays(fashion["train"], N_TRAIN)
Xte, yte = to_arrays(fashion["test"], N_TEST)

SHADE, WIDTH = ["dark", "bright"], ["narrow", "wide"]

def attributes(X):
    return X.reshape(len(X), -1).mean(1), (X.max(axis=1) > 0.25).sum(axis=1)

ink_thr, col_thr = (lambda a: (np.median(a[0]), np.median(a[1])))(attributes(Xtr))

def captions(X, y):
    ink, col = attributes(X)
    s, w = (ink > ink_thr).astype(int), (col > col_thr).astype(int)
    return [f"a {SHADE[a]} {WIDTH[b]} photo of a {CLASS_NAMES[c]}" for a, b, c in zip(s, w, y)], s, w

cap_tr, s_tr, w_tr = captions(Xtr, ytr)
cap_te, s_te, w_te = captions(Xte, yte)

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 16                                # <- knob
enc_tr = tok(cap_tr, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
img_tr = torch.from_numpy(Xtr).unsqueeze(1)
img_te = torch.from_numpy(Xte).unsqueeze(1)

print(cap_tr[0], "|", cap_tr[1])
print("images:", tuple(img_tr.shape), "| captions:", tuple(enc_tr["input_ids"].shape))

## 1.2 The vision tower, recovered from Chapter 13

💻 **The code.** Chapter 13's CLIP, retrained here in ten seconds so this
notebook stands alone. Two things differ from that chapter's version, and both
matter for what follows:

- `tokens()` returns **all** patch tokens, not just `[CLS]`. A classifier only
  needs the pooled summary; a captioner may want spatial detail, and Module 5
  tests whether it actually does.
- Once trained, every parameter is frozen. From here on the vision tower is a
  fixed function, and it is not learning anything about captions.


In [ ]:
D_EMB = 128


class Block(nn.Module):
    '''Chapter 10's pre-LN block. `causal` adds Chapter 9's triangular mask.'''

    def __init__(self, d, h, ratio=4):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, ratio * d), nn.GELU(), nn.Linear(ratio * d, d))

    def forward(self, x, pad_mask=None, causal=False):
        a = self.n1(x)
        mask = None
        if causal:
            n = x.shape[1]
            mask = torch.triu(torch.ones(n, n, device=x.device, dtype=torch.bool), 1)
        x = x + self.attn(a, a, a, key_padding_mask=pad_mask, attn_mask=mask, need_weights=False)[0]
        return x + self.mlp(self.n2(x))


class ImageEncoder(nn.Module):
    def __init__(self, d=128, patch=4, depth=4, heads=4, out_dim=D_EMB):
        super().__init__()
        self.patch = nn.Conv2d(1, d, patch, patch)
        n = (28 // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.proj = nn.Linear(d, out_dim, bias=False)

    def tokens(self, x):
        '''All 1 + 49 tokens, not just the pooled one.'''
        x = self.patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1) + self.pos
        for b in self.blocks:
            x = b(x)
        return self.norm(x)

    def forward(self, x):
        return self.proj(self.tokens(x)[:, 0])


class TextEncoder(nn.Module):
    def __init__(self, vocab, d=128, depth=4, heads=4, max_len=MAX_LEN, out_dim=D_EMB):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.proj = nn.Linear(d, out_dim, bias=False)

    def forward(self, ids, mask):
        x = self.emb(ids) + self.pos[:, : ids.shape[1]]
        for b in self.blocks:
            x = b(x, pad_mask=(mask == 0))
        x = self.norm(x)
        return self.proj(x[torch.arange(len(x)), mask.sum(1) - 1])


class CLIP(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.image, self.text = ImageEncoder(), TextEncoder(vocab)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, im, ids, mask):
        zi = F.normalize(self.image(im), dim=-1)
        zt = F.normalize(self.text(ids, mask), dim=-1)
        return self.logit_scale.exp().clamp(max=100) * zi @ zt.T


def clip_loss(logits):
    t = torch.arange(len(logits), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, t) + F.cross_entropy(logits.T, t))


def pretrain_clip(epochs=8, batch=256, seed=0):        # <- knob
    torch.manual_seed(seed)
    m = CLIP(tok.vocab_size).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.1)
    ids, mask = enc_tr["input_ids"], enc_tr["attention_mask"]
    for _ in range(epochs):
        perm = torch.randperm(len(img_tr))
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i:i + batch]
            loss = clip_loss(m(img_tr[j].to(device), ids[j].to(device), mask[j].to(device)))
            opt.zero_grad(); loss.backward(); opt.step()
    return m


t0 = time.time()
vision = pretrain_clip().image.eval()
for p in vision.parameters():
    p.requires_grad = False                            # frozen from here on
print(f"vision tower pretrained and frozen in {time.time() - t0:.1f}s "
      f"({sum(p.numel() for p in vision.parameters()):,} frozen parameters)")

---
# Module 2 — The Language Model

🧠 **The intuition.** Every attention model you have built so far has been an
*encoder*: every token sees every other token, which is right for
classification and wrong for generation. To produce text one token at a time,
the model must never see the future, which is the causal mask from Chapter 9,
Module 5, and it is the only architectural difference. Same blocks, same
attention, one triangular matrix.

📐 **The math.** An autoregressive model factorizes the caption's probability:

$$ p(t_1, \dots, t_m) = \prod_{i=1}^{m} p(t_i \mid t_{<i}) $$

and is trained by maximising the log of that, meaning cross-entropy on every
position at once, which is efficient precisely *because* the causal mask lets
one forward pass score all $m$ predictions without leaking answers.

**We tie the output head to the embedding table** (`head.weight = emb.weight`).
With a 30,522-token vocabulary and $d = 128$ that saves 3.9M parameters, and it
encodes a sensible prior: the vector you use to *represent* a word is a
reasonable vector to *score* it with.

💻 **The code.**


In [ ]:
D_LM = 128


class DecoderLM(nn.Module):
    '''Chapter 10's block + Chapter 9's causal mask = a language model.'''

    def __init__(self, vocab, d=D_LM, depth=4, heads=4, max_len=MAX_LEN + 8):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.emb.weight            # tied: same matrix in and out

    def forward_embeds(self, e):
        '''Takes embeddings, not ids, which is what lets images enter the sequence.'''
        x = e + self.pos[:, : e.shape[1]]
        for b in self.blocks:
            x = b(x, causal=True)
        return self.head(self.norm(x))


def pretrain_lm(epochs=8, batch=256, lr=3e-4, seed=0):        # <- knob
    '''Stage 0: learn the language, with no images anywhere.'''
    torch.manual_seed(seed)
    lm = DecoderLM(tok.vocab_size).to(device)
    opt = torch.optim.AdamW(lm.parameters(), lr=lr, weight_decay=0.01)
    ids = enc_tr["input_ids"]
    hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(ids)); tot, nb = 0.0, 0
        for i in range(0, len(perm) - batch + 1, batch):
            b = ids[perm[i:i + batch]].to(device)
            lg = lm.forward_embeds(lm.emb(b))
            loss = F.cross_entropy(lg[:, :-1].reshape(-1, lg.shape[-1]),
                                   b[:, 1:].reshape(-1), ignore_index=tok.pad_token_id)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / nb)
    return lm, hist


t0 = time.time()
base_lm, lm_hist = pretrain_lm()
print(f"text-only LM pretrained in {time.time() - t0:.1f}s | final loss {lm_hist[-1]:.4f}")
print(f"parameters: {sum(p.numel() for p in base_lm.parameters()):,}")

This stand-in for "a pretrained LLM" is doing the same job GPT or Llama does in
a real VLM, at a scale that fits in thirty seconds: it has learned the *shape*
of the sentences (that they open with "a", that a shade word comes next, that a
garment name ends them) without any idea which garment. Module 5 shows that
this, and not the projector, is where most of the competence lives.


---
# Module 3 — The Projector

🧠 **The intuition.** The vision tower emits 128-dimensional feature vectors.
The language model expects 128-dimensional *embedding* vectors. Those numbers
matching is a coincidence of our configuration and means nothing, since the two
spaces have completely unrelated geometry. The projector's job is to learn the
translation.

Which image features should become tokens? We take four: the pooled `[CLS]`
summary, the mean over all patches, and two individual patch tokens. In real
LLaVA this is *all* 576 patch tokens, which is why LLaVA's sequences are long
and why later work (Q-Former, perceiver resamplers) spends effort compressing
them back down. Module 5 measures whether four is better than one here.

📐 **The math.** $P_\theta : \mathbb{R}^{d_{\text{img}}} \to \mathbb{R}^{d_{\text{lm}}}$,
applied independently to each selected feature vector. LLaVA v1 used a single
`Linear`; v1.5 found a two-layer MLP measurably better, so that is what we use:

$$ P_\theta(v) = W_2\,\mathrm{GELU}(W_1 v) $$

That is the *entire* set of new parameters in the model. Everything else is
either frozen or pretrained.

💻 **The code.** Note `forward_embeds` in the LM above: the projector's output
is concatenated with word embeddings and the LM never learns which was which.


In [ ]:
class VLM(nn.Module):
    '''LLaVA in miniature: frozen vision tower -> projector -> prefix tokens -> LM.'''

    def __init__(self, vision, vocab, n_prefix=4, d_img=128, d_lm=D_LM, blind=False):
        super().__init__()
        self.vision, self.n_prefix, self.blind = vision, n_prefix, blind
        self.projector = nn.Sequential(nn.Linear(d_img, d_lm), nn.GELU(), nn.Linear(d_lm, d_lm))
        self.lm = DecoderLM(vocab, d=d_lm)

    def prefix(self, imgs):
        with torch.no_grad():                       # the tower is frozen: no gradient needed
            t = self.vision.tokens(imgs)            # (B, 1 + 49, d)
        take = torch.cat([t[:, :1],                 # [CLS] summary
                          t[:, 1:].mean(1, keepdim=True),   # mean over patches
                          t[:, 1:3]], 1)            # two individual patches
        return self.projector(take[:, : self.n_prefix])

    def forward(self, imgs, ids):
        e = self.lm.emb(ids)
        if self.blind:
            return self.lm.forward_embeds(e), 0
        p = self.prefix(imgs)
        return self.lm.forward_embeds(torch.cat([p, e], 1)), p.shape[1]


def lm_loss(logits, ids, k, pad_id):
    '''Cross-entropy on TEXT positions only: image tokens are context, not targets.'''
    logits = logits[:, k:-1]
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                           ids[:, 1:].reshape(-1), ignore_index=pad_id)


demo = VLM(vision, tok.vocab_size).to(device)
print(f"projector parameters (the only genuinely new weights): "
      f"{sum(p.numel() for p in demo.projector.parameters()):,}")
print(f"frozen vision tower: {sum(p.numel() for p in demo.vision.parameters()):,}")
print(f"language model:      {sum(p.numel() for p in demo.lm.parameters()):,}")

---
# Module 4 — Training, Generating, and the Baseline That Keeps Us Honest

## 4.1 Train and generate

💻 **The code.** Greedy decoding: start from `[CLS]`, take the argmax at each
step, append, repeat. The image tokens sit in front of the sequence the whole
time and are recomputed once.

In [ ]:
def train_vlm(pretrained_lm=None, freeze_lm=False, n_prefix=4, blind=False,
              epochs=8, batch=256, lr=3e-4, seed=0):        # <- knobs
    torch.manual_seed(seed)
    m = VLM(vision, tok.vocab_size, n_prefix=n_prefix, blind=blind).to(device)
    if pretrained_lm is not None:
        m.lm.load_state_dict(pretrained_lm.state_dict())
    if freeze_lm:
        for p in m.lm.parameters():
            p.requires_grad = False
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    ids = enc_tr["input_ids"]
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(img_tr))
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i:i + batch]
            lg, k = m(img_tr[j].to(device), ids[j].to(device))
            loss = lm_loss(lg, ids[j].to(device), k, tok.pad_token_id)
            opt.zero_grad(); loss.backward(); opt.step()
    return m, sum(p.numel() for p in params)


@torch.no_grad()
def generate(m, imgs, max_new=12):
    m.eval()
    ids = torch.full((len(imgs), 1), tok.cls_token_id, device=device, dtype=torch.long)
    p = None if m.blind else m.prefix(imgs.to(device))
    for _ in range(max_new):
        e = m.lm.emb(ids)
        e = e if m.blind else torch.cat([p, e], 1)
        ids = torch.cat([ids, m.lm.forward_embeds(e)[:, -1].argmax(-1, keepdim=True)], 1)
    return tok.batch_decode(ids[:, 1:], skip_special_tokens=True)


t0 = time.time()
vlm, n_trained = train_vlm(pretrained_lm=base_lm)
print(f"trained in {time.time() - t0:.1f}s | {n_trained:,} parameters updated\n")
for text, y in list(zip(generate(vlm, img_te[:6]), yte))[:6]:
    print(f"  generated: {text:<45} true class: {CLASS_NAMES[y]}")

In [ ]:
idx = np.arange(8) * 137
texts = generate(vlm, img_te[idx])
fig, axes = plt.subplots(1, 8, figsize=(14, 2.4))
for ax, i, t in zip(axes, idx, texts):
    ax.imshow(Xte[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title("\n".join(t.split(" photo of a ")), fontsize=6.5)
    ax.axis("off")
fig.suptitle("Captions generated one token at a time, from four image tokens", y=1.08)
plt.tight_layout()
plt.show()

## 4.2 The baseline: does it use the image at all?

🧠 **The intuition, and the trap.** Those captions look right. But our caption
distribution is narrow (39 distinct sentences), and a language model that
ignored the image entirely could still produce fluent, plausible, grammatical
captions by sampling the prior. Fluency is not evidence of grounding.

So we build the **blind** model: identical in every respect, trained the same
way, with the image tokens removed. Whatever it scores is what "sounding right"
is worth on its own, and only the margin above that belongs to vision.

📐 **How we score.** Parse the generated string for a class name, a shade word
and a width word, and compare each against the ground truth for that image.
Three separate accuracies rather than one, because they can fail independently,
and the two attribute scores are 2-way (chance 0.5) while class is 10-way
(chance 0.1).

💻 **The code.**


In [ ]:
def score(texts, n=None):
    n = n or len(texts)
    cls = np.array([next((i for i, c in enumerate(CLASS_NAMES) if c.lower() in t.lower()), -1)
                    for t in texts])
    sh = np.array([1 if "bright" in t else (0 if "dark" in t else -1) for t in texts])
    wd = np.array([1 if "wide" in t else (0 if "narrow" in t else -1) for t in texts])
    return (cls == yte[:n]).mean(), (sh == s_te[:n]).mean(), (wd == w_te[:n]).mean()


N_EVAL = 2000                                   # <- knob
blind, _ = train_vlm(pretrained_lm=base_lm, blind=True)

rows = [("VLM  (4 image tokens)", score(generate(vlm, img_te[:N_EVAL]), N_EVAL)),
        ("blind LM (image removed)", score(generate(blind, img_te[:N_EVAL]), N_EVAL))]

print(f"{'model':<28}{'class':>8}{'shade':>8}{'width':>8}")
for name, (c, s, w) in rows:
    print(f"{name:<28}{c:>8.3f}{s:>8.3f}{w:>8.3f}")
print(f"{'chance':<28}{0.1:>8.3f}{0.5:>8.3f}{0.5:>8.3f}")
print("\nblind sample:", generate(blind, img_te[:1])[0])

**The blind model lands on chance for all three**, and its output is the same
sentence every time, the single most likely caption, since with no image there
is nothing to distinguish one input from another. That is exactly what a caption
prior looks like, and it is what our grounded numbers have to be read against.

Keep this baseline in mind whenever you see a VLM demo. A model that describes a
photo of a beach as "a beautiful sunset over the ocean" may be looking, or may
be reciting; the caption alone cannot tell you which, and the field has
repeatedly found real systems doing more of the latter than advertised.


---
# Module 5 — The LLaVA Ablation

🧠 **The claim under test.** LLaVA's headline is that stage 1 trains *only* the
projector, with the vision tower frozen and the language model frozen, and that
this is enough to produce a working vision-language model. If true, a
33,000-parameter MLP is doing something remarkable.

📐 **What to vary.** Three conditions, one question each:

| Condition | Question |
|---|---|
| random LM, frozen, projector only | Can the projector do the job by itself? |
| **pretrained** LM, frozen, projector only | Is LLaVA's stage 1 real? |
| pretrained LM, unfrozen | What does stage 2 add? |

and separately, `n_prefix = 1` vs `4`, because "how many image tokens" is the
design decision every VLM paper argues about.

💻 **The code.** Four more training runs; this is the slow cell.


In [ ]:
def run(label, **kw):
    m, n = train_vlm(**kw)
    c, s, w = score(generate(m, img_te[:N_EVAL]), N_EVAL)
    print(f"{label:<40}{n:>12,}{c:>8.3f}{s:>8.3f}{w:>8.3f}")
    return m


print(f"{'condition':<40}{'trained':>12}{'class':>8}{'shade':>8}{'width':>8}")
run("random LM, frozen (projector only)",     pretrained_lm=None,    freeze_lm=True)
run("pretrained LM, frozen (LLaVA stage 1)",  pretrained_lm=base_lm, freeze_lm=True)
run("pretrained LM, unfrozen (stage 2)",      pretrained_lm=base_lm, freeze_lm=False)
print()
run("1 image token,  pretrained + unfrozen",  pretrained_lm=base_lm, n_prefix=1)
run("4 image tokens, pretrained + unfrozen",  pretrained_lm=base_lm, n_prefix=4)

## 5.1 Reading the ablation

**Three results, and the middle one is the chapter.**

**A frozen *random* language model cannot be rescued by a projector.** It scores
zero on everything, not "poor" but zero, because it never learned that captions
are English and the projector has no way to teach it. This is the control that
makes the next row mean something.

**A frozen *pretrained* language model works, and works as well as fine-tuning
the whole thing.** Roughly 33,000 trained parameters match roughly 4.7 million,
a **~140×** reduction for no measurable loss. That is LLaVA's claim, reproduced.
The projector is not learning to caption; it is learning a *change of
coordinates* into a space where a model that already knows how to caption can
read the image. All the competence was already present in the two frozen towers,
and the missing piece was genuinely just an adapter.

**Stage 2 adds nothing measurable here**, which is an honest negative and worth
being careful about. Real LLaVA's stage 2 exists to teach *instruction
following* (answering questions, holding a conversation, refusing), and our
task is a single fixed caption format with no instructions in it at all. We have
built the situation in which stage 2 has nothing to contribute, so this
measurement should not be read as evidence against it.

**One image token performs the same as four.** Also honest, also expected once
you look at the task: Fashion-MNIST garments are centred, single-object, and
28×28, so the pooled summary already carries everything the caption needs.
Spatial tokens pay off when the question is *where*: counting objects,
reading text in an image, describing relations between things, and none of our
captions ask that. It does illustrate the real trade: LLaVA's 576 tokens per
image dominate its context budget, and whether that is worth it is entirely a
function of what you are asking.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| Frozen vision tower + projector + LM | A language model's layers only see vectors. Anything you can map into embedding space becomes a token, including a picture |
| `forward_embeds` instead of `forward(ids)` | The one line that makes multimodality possible: accept embeddings, not token ids |
| Loss on text positions only | Image tokens are context, never prediction targets |
| Greedy decoding with a prefix | Generation is the causal mask from Chapter 9 plus a loop |
| The blind baseline | Fluent captions are not evidence of grounding. Always build the model that cannot see |
| The frozen-LM ablation | 33k trained parameters match 4.7M, but only because the LM was already pretrained. The random-LM control is what proves it |
| 1 vs 4 image tokens | No difference on centred single-object images. Spatial tokens pay off for *where* questions, and cost context budget |

**Where the pieces came from.** This chapter added almost no new machinery. The
vision tower is Chapter 11's ViT trained by Chapter 13's contrastive loss; the
language model is Chapter 10's encoder block with Chapter 9's causal mask; the
only genuinely new object is a two-layer MLP. That is worth noticing as a
pattern in itself: a great deal of "multimodal AI" is adapters between
pretrained components, and the adapters are small.

**The thread this leaves open.** Everything from Chapter 9 onwards has been
built on attention, and attention has been paying the same $O(n^2)$ bill since
we first measured it. Prefix tokens make the bill worse: LLaVA's 576 image
tokens are 576 positions every text token must attend to, at every layer. That
cost is the reason context length is the number everyone quotes, and the reason
a serious line of research asks whether the quadratic term is necessary at all.
